# SCC0276 - Aprendizado de Máquina - EXERCÍCIO 2

1. Nome aluno - NUSP
2. Nome aluno - NUSP
...


## Carregando dados

Para este exercício, será utilizado o conjunto de dados *Breast Cancer* disponibilizado pelo [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).

In [1]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

In [2]:
raw_data = load_breast_cancer()

In [3]:
df = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)
labels = raw_data.target

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

## 1) Perceptron (regressão logística) e avaliação de performance

Neste exercício, o objetivo é investigar o comportamento do modelo Perceptron, suas limitações e avaliá-lo em um exemplo de problema de classificação. Além disto, serão abordadas duas estratégias diferentes de avaliação: *holdout* e *K-Folds*.

- a) Divida o os dados em treino e teste usando a função [*train_test_split*](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) do sklearn. Esta função implementa a avaliação baseada em *holdout*. Utilize 20% dos dados para avaliação, e a seed 2026 para a divisão dos dados. Estratifique os conjuntos com base nos rótulos (veja na documentação o significado do parâmetro '*stratify*').
- b) Padronize os dados como pré-processamento.
- c) Compute a acurácia no conjunto de teste e a medida revocação, utilizando o [Perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html) com seus parâmetros padrão. Apenas configure sua *seed* para a estabelecida na divisão de treino e teste.
- d) Repita os itens anteriores, utilizando uma transformação polinomial das *features* originais. Utilize o [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html#sklearn.preprocessing.PolynomialFeatures) (usando "*degree*" igual a 2) como pré-processamento no lugar do StandardScaler.
- e) Repita a avaliação usando uma validação cruzada de 5-folds estratificada (use [K-folds](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) ou [*cross_val_score*](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)).  **Para simplificar o exercício, utilize os dados sem pré-processamento.**


In [5]:
# =============================
# Exercício 1a) Divisão treino/teste (Holdout)
# =============================

from sklearn.model_selection import train_test_split

SEED = 2026

# Divide os dados: 80% treino, 20% teste, estratificado pelos rótulos
X_train, X_test, y_train, y_test = train_test_split(
    df, labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels
)

print(f"Tamanho do conjunto de treino: {X_train.shape[0]}")
print(f"Tamanho do conjunto de teste:  {X_test.shape[0]}")
print(f"Proporção das classes no treino: {pd.Series(y_train).value_counts(normalize=True).to_dict()}")
print(f"Proporção das classes no teste:  {pd.Series(y_test).value_counts(normalize=True).to_dict()}")

Tamanho do conjunto de treino: 455
Tamanho do conjunto de teste:  114
Proporção das classes no treino: {1: 0.6263736263736264, 0: 0.37362637362637363}
Proporção das classes no teste:  {1: 0.631578947368421, 0: 0.3684210526315789}


In [6]:
# =============================
# Exercício 1b) Padronização dos dados (StandardScaler)
# =============================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit no treino e transforma treino e teste
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dados padronizados com sucesso!")
print(f"Média das features no treino (aprox. 0): {X_train_scaled.mean(axis=0)[:3].round(6)} ...")
print(f"Desvio padrão no treino (aprox. 1):      {X_train_scaled.std(axis=0)[:3].round(6)} ...")

Dados padronizados com sucesso!
Média das features no treino (aprox. 0): [ 0. -0. -0.] ...
Desvio padrão no treino (aprox. 1):      [1. 1. 1.] ...


In [7]:
# =============================
# Exercício 1c) Perceptron com StandardScaler - Acurácia e Recall
# =============================

from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, recall_score

# Cria o Perceptron com parâmetros padrão e seed definida
perceptron = Perceptron(random_state=SEED)

# Treina com dados padronizados
perceptron.fit(X_train_scaled, y_train)

# Predições no conjunto de teste
y_pred_perc = perceptron.predict(X_test_scaled)

# Métricas
acc_perc = accuracy_score(y_test, y_pred_perc)
rec_perc = recall_score(y_test, y_pred_perc)

print("=== Perceptron com StandardScaler ===")
print(f"Acurácia:  {acc_perc:.4f}")
print(f"Revocação: {rec_perc:.4f}")

=== Perceptron com StandardScaler ===
Acurácia:  0.9737
Revocação: 0.9861


In [8]:
# =============================
# Exercício 1d) Perceptron com PolynomialFeatures (degree=2)
# =============================

from sklearn.preprocessing import PolynomialFeatures

# Cria a transformação polinomial de grau 2
poly = PolynomialFeatures(degree=2)

# Transforma os dados (fit no treino, transforma treino e teste)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f"Features originais: {X_train.shape[1]}")
print(f"Features polinomiais: {X_train_poly.shape[1]}")

# Treina o Perceptron com dados polinomiais
perceptron_poly = Perceptron(random_state=SEED)
perceptron_poly.fit(X_train_poly, y_train)

# Predições
y_pred_poly = perceptron_poly.predict(X_test_poly)

# Métricas
acc_poly = accuracy_score(y_test, y_pred_poly)
rec_poly = recall_score(y_test, y_pred_poly)

print(f"\n=== Perceptron com PolynomialFeatures (degree=2) ===")
print(f"Acurácia:  {acc_poly:.4f}")
print(f"Revocação: {rec_poly:.4f}")

# Comparação
print(f"\n=== Comparação ===")
print(f"{'Modelo':<35} {'Acurácia':>10} {'Revocação':>10}")
print(f"{'Perceptron + StandardScaler':<35} {acc_perc:>10.4f} {rec_perc:>10.4f}")
print(f"{'Perceptron + PolynomialFeatures':<35} {acc_poly:>10.4f} {rec_poly:>10.4f}")

Features originais: 30
Features polinomiais: 496

=== Perceptron com PolynomialFeatures (degree=2) ===
Acurácia:  0.9474
Revocação: 0.9583

=== Comparação ===
Modelo                                Acurácia  Revocação
Perceptron + StandardScaler             0.9737     0.9861
Perceptron + PolynomialFeatures         0.9474     0.9583


In [9]:
# =============================
# Exercício 1e) Validação cruzada 5-folds estratificada (sem pré-processamento)
# =============================

from sklearn.model_selection import cross_val_score, StratifiedKFold

# Configura a validação cruzada estratificada com 5 folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Perceptron sem pré-processamento
perceptron_cv = Perceptron(random_state=SEED)

# Acurácia com cross_val_score (dados originais, sem pré-processamento)
scores_acc = cross_val_score(perceptron_cv, df, labels, cv=skf, scoring='accuracy')
scores_rec = cross_val_score(perceptron_cv, df, labels, cv=skf, scoring='recall')

print("=== Perceptron - Validação Cruzada 5-Folds (sem pré-processamento) ===")
print(f"Acurácias por fold:  {scores_acc.round(4)}")
print(f"Acurácia média:      {scores_acc.mean():.4f} ± {scores_acc.std():.4f}")
print(f"\nRevocações por fold: {scores_rec.round(4)}")
print(f"Revocação média:     {scores_rec.mean():.4f} ± {scores_rec.std():.4f}")

=== Perceptron - Validação Cruzada 5-Folds (sem pré-processamento) ===
Acurácias por fold:  [0.9298 0.8158 0.8596 0.9386 0.7257]
Acurácia média:      0.8539 ± 0.0786

Revocações por fold: [0.9577 1.     0.9861 1.     1.    ]
Revocação média:     0.9888 ± 0.0164


## 2) Multi-Layer Perceptron

Dando continuidade, este exercício foca na estudo do modelo MLP. 

- a) Repita os 3 primeiros itens do exercício anterior, avaliando uma MLP no lugar do Perceptron. Utilize os parâmetros padrão da [MLP](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) do sklearn inicialmente e a *seed* igual a 2026.
- b) Avalie a MLP, usando a validação *holdout* dos exercícios anteriores, utilizando 3 configurações de parâmetos diferentes. Dê preferência para modificar número de neurônios e camadas.
- c) Compare a melhor MLP deste exercício, com o melhor Perceptron do anterior utilizando uma validação cruzada estratificada de 5-folds. **Por simplicidade, utilize dados sem pré-processamento.**


In [10]:
# =============================
# Exercício 2a) MLP com parâmetros padrão + StandardScaler (Holdout)
# =============================

from sklearn.neural_network import MLPClassifier

# Reutiliza a mesma divisão treino/teste e os dados já padronizados (StandardScaler)
# do Exercício 1

# Cria a MLP com parâmetros padrão e seed 2026
mlp = MLPClassifier(random_state=SEED)

# Treina com dados padronizados
mlp.fit(X_train_scaled, y_train)

# Predições
y_pred_mlp = mlp.predict(X_test_scaled)

# Métricas
acc_mlp = accuracy_score(y_test, y_pred_mlp)
rec_mlp = recall_score(y_test, y_pred_mlp)

print("=== MLP (parâmetros padrão) com StandardScaler ===")
print(f"Acurácia:  {acc_mlp:.4f}")
print(f"Revocação: {rec_mlp:.4f}")

=== MLP (parâmetros padrão) com StandardScaler ===
Acurácia:  0.9737
Revocação: 0.9861


/home/tur1sm0/anaconda3/envs/AM/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [11]:
# =============================
# Exercício 2b) MLP com 3 configurações diferentes de parâmetros
# =============================

# Configuração 1: 1 camada com 50 neurônios
mlp_config1 = MLPClassifier(hidden_layer_sizes=(50,), random_state=SEED)
mlp_config1.fit(X_train_scaled, y_train)
y_pred_c1 = mlp_config1.predict(X_test_scaled)
acc_c1 = accuracy_score(y_test, y_pred_c1)
rec_c1 = recall_score(y_test, y_pred_c1)

# Configuração 2: 2 camadas com 100 e 50 neurônios
mlp_config2 = MLPClassifier(hidden_layer_sizes=(100, 50), random_state=SEED)
mlp_config2.fit(X_train_scaled, y_train)
y_pred_c2 = mlp_config2.predict(X_test_scaled)
acc_c2 = accuracy_score(y_test, y_pred_c2)
rec_c2 = recall_score(y_test, y_pred_c2)

# Configuração 3: 3 camadas com 128, 64 e 32 neurônios
mlp_config3 = MLPClassifier(hidden_layer_sizes=(128, 64, 32), random_state=SEED)
mlp_config3.fit(X_train_scaled, y_train)
y_pred_c3 = mlp_config3.predict(X_test_scaled)
acc_c3 = accuracy_score(y_test, y_pred_c3)
rec_c3 = recall_score(y_test, y_pred_c3)

# Resumo das configurações
print("=== Comparação de MLPs com diferentes configurações ===")
print(f"{'Configuração':<35} {'Acurácia':>10} {'Revocação':>10}")
print("-" * 57)
print(f"{'MLP padrão (100,)':<35} {acc_mlp:>10.4f} {rec_mlp:>10.4f}")
print(f"{'MLP (50,)':<35} {acc_c1:>10.4f} {rec_c1:>10.4f}")
print(f"{'MLP (100, 50)':<35} {acc_c2:>10.4f} {rec_c2:>10.4f}")
print(f"{'MLP (128, 64, 32)':<35} {acc_c3:>10.4f} {rec_c3:>10.4f}")

# Identifica a melhor configuração
configs = {
    'MLP padrão (100,)':   (acc_mlp, rec_mlp),
    'MLP (50,)':           (acc_c1, rec_c1),
    'MLP (100, 50)':       (acc_c2, rec_c2),
    'MLP (128, 64, 32)':   (acc_c3, rec_c3),
}
melhor_mlp = max(configs, key=lambda k: configs[k][0])
print(f"\nMelhor MLP (por acurácia): {melhor_mlp} - Acurácia: {configs[melhor_mlp][0]:.4f}")

/home/tur1sm0/anaconda3/envs/AM/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


=== Comparação de MLPs com diferentes configurações ===
Configuração                          Acurácia  Revocação
---------------------------------------------------------
MLP padrão (100,)                       0.9737     0.9861
MLP (50,)                               0.9649     0.9722
MLP (100, 50)                           0.9825     0.9861
MLP (128, 64, 32)                       0.9737     0.9861

Melhor MLP (por acurácia): MLP (100, 50) - Acurácia: 0.9825


In [12]:
# =============================
# Exercício 2c) Comparação melhor MLP vs. melhor Perceptron (5-Folds, sem pré-proc.)
# =============================

# Validação cruzada estratificada de 5-folds, sem pré-processamento
# Reutiliza o StratifiedKFold já configurado (skf)

# Melhor Perceptron: com StandardScaler foi o melhor no holdout
perceptron_cv2 = Perceptron(random_state=SEED)
scores_perc_acc = cross_val_score(perceptron_cv2, df, labels, cv=skf, scoring='accuracy')
scores_perc_rec = cross_val_score(perceptron_cv2, df, labels, cv=skf, scoring='recall')

# Melhor MLP: usamos a config padrão como baseline (ajustar conforme resultado do 2b)
mlp_cv = MLPClassifier(random_state=SEED)
scores_mlp_acc = cross_val_score(mlp_cv, df, labels, cv=skf, scoring='accuracy')
scores_mlp_rec = cross_val_score(mlp_cv, df, labels, cv=skf, scoring='recall')

print("=== Comparação via Validação Cruzada 5-Folds (sem pré-processamento) ===\n")

print("Perceptron:")
print(f"  Acurácias por fold:  {scores_perc_acc.round(4)}")
print(f"  Acurácia média:      {scores_perc_acc.mean():.4f} ± {scores_perc_acc.std():.4f}")
print(f"  Revocação média:     {scores_perc_rec.mean():.4f} ± {scores_perc_rec.std():.4f}")

print(f"\nMLP (parâmetros padrão):")
print(f"  Acurácias por fold:  {scores_mlp_acc.round(4)}")
print(f"  Acurácia média:      {scores_mlp_acc.mean():.4f} ± {scores_mlp_acc.std():.4f}")
print(f"  Revocação média:     {scores_mlp_rec.mean():.4f} ± {scores_mlp_rec.std():.4f}")

=== Comparação via Validação Cruzada 5-Folds (sem pré-processamento) ===

Perceptron:
  Acurácias por fold:  [0.9298 0.8158 0.8596 0.9386 0.7257]
  Acurácia média:      0.8539 ± 0.0786
  Revocação média:     0.9888 ± 0.0164

MLP (parâmetros padrão):
  Acurácias por fold:  [0.9386 0.8947 0.9211 0.9649 0.9558]
  Acurácia média:      0.9350 ± 0.0251
  Revocação média:     0.9691 ± 0.0207


## 3) Seleção de parâmetros

Note que no exercício anterior, foram testados diferentes configurações de MLP de forma manual. Neste exercício, o foco será em testar diferentes configurações do modelo de forma automática e reportar a melhor configuração.

- a) Leia a documentação do [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).
- b) Configure o "*param_grid*" com ao menos dois parâmetros (Ex: '*hidden_layer_sizes*' e '*activation*').
- c) Escolha uma métrica de classificação para comparar as MLPs e execute o GridSearchCV. **Para simplificar o exercício, utilize os dados sem pré-processamento no 'fit'.**
- d) Indique a melhor configuração obtida em termos da métrica escolhida.

In [13]:
# =============================
# Exercício 3a-d) GridSearchCV para seleção de parâmetros da MLP
# =============================

from sklearn.model_selection import GridSearchCV

# b) Definindo o grid de parâmetros: hidden_layer_sizes e activation
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (128, 64, 32)],
    'activation':         ['relu', 'tanh', 'logistic']
}

# c) Métrica escolhida: acurácia (accuracy)
# Usa StratifiedKFold de 5-folds como validação interna
grid_search = GridSearchCV(
    estimator=MLPClassifier(random_state=SEED, max_iter=500),
    param_grid=param_grid,
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    refit=True,
    n_jobs=-1
)

# Treina o GridSearch sem pré-processamento (conforme enunciado)
grid_search.fit(df, labels)

# d) Melhor configuração e melhor score
print("=== Resultados do GridSearchCV ===\n")
print(f"Melhor acurácia (CV): {grid_search.best_score_:.4f}")
print(f"Melhores parâmetros:  {grid_search.best_params_}")

# Tabela com todos os resultados
import numpy as np
results = pd.DataFrame(grid_search.cv_results_)
cols = ['param_hidden_layer_sizes', 'param_activation', 'mean_test_score', 'std_test_score', 'rank_test_score']
print(f"\n=== Ranking completo ===")
print(results[cols].sort_values('rank_test_score').to_string(index=False))

=== Resultados do GridSearchCV ===

Melhor acurácia (CV): 0.9438
Melhores parâmetros:  {'activation': 'logistic', 'hidden_layer_sizes': (50,)}

=== Ranking completo ===
param_hidden_layer_sizes param_activation  mean_test_score  std_test_score  rank_test_score
                   (50,)         logistic         0.943798        0.020396                1
               (100, 50)         logistic         0.940289        0.020234                2
                  (100,)             relu         0.935010        0.025116                3
                  (100,)         logistic         0.933256        0.012994                4
                   (50,)             tanh         0.933240        0.017991                5
           (128, 64, 32)         logistic         0.927977        0.024363                6
                  (100,)             tanh         0.927977        0.020221                6
               (100, 50)             tanh         0.927961        0.017829                8
   

## Questões teóricas para estudo

- Em problemas de classificação, por que é importante avaliar outras métricas além de acurácia?
- Qual a diferença entre a validação *holdout* e a validação cruzada via K-folds? Aponte uma vantagem da última técnica.
- Note que o Percetron e a MLP foram avaliados utilizando os mesmos conjuntos de treino e teste. Discuta por que isto é importante.
- Ao usar uma validação cruzada baseada em K-folds, é preciso refazer o pré-processamento dos dados em cada *fold*. Por que isso deve ser feito?
- Investigue como adicionar pre-processamento ao GridSearchCV. Dica: [Pipelines](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).  
- O que o [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html#sklearn.preprocessing.PolynomialFeatures) faz como transformação? O Perceptron, em conjunto deste pré-processamento, ainda é um classificador linear?